In [51]:
import torch

In [52]:
import torch

x = torch.zeros(2, 3, 4)

print(x.shape)
print(x.ndim)
print(x.numel())

torch.Size([2, 3, 4])
3
24


In [53]:
# Question 2 — Broadcasting
# Predict the output before running the code.

a = torch.ones(2, 3)
b = torch.tensor([1., 2., 3.])

print(a + b)

tensor([[2., 3., 4.],
        [2., 3., 4.]])


In [54]:
# Question 3 — Autograd and derivatives
# This is one of the most important PyTorch concepts for deep learning.

# Given: 

#                     y = 3x^2 + 2x

# Calculate \(\frac{dy}{dx}\) at \(x=2\), then verify it using autograd.


x = torch.tensor(2.0, requires_grad= True)

y = 3 * x**2 + 2 * x

y.backward()
print(x.grad)


tensor(14.)


In [55]:
# Question 4 — Gradient accumulation
# What happens if you call backward() repeatedly without clearing the gradients?

x = torch.tensor(3.0, requires_grad=True)

(x**2).backward()
(x**2).backward()

print(x.grad)

# Output: tensor(12.).

# Each backward pass contributes a gradient of \(2x=6\), so the accumulated gradient is \(6+6=12\).

# In a normal training loop, use optimizer.zero_grad() before computing the next update.

tensor(12.)


In [56]:
# Question 5 — Build a small neural network
# Create a model that accepts 10 input features and predicts one of 3 classes.

# Your task:
# - Use nn.Linear.
# - Add a ReLU activation.
# - Return 3 output logits.
# - Test it with a batch of 8 examples.

import torch
import torch.nn as nn

class Classifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer = nn.Sequential(
            nn.Linear(10,32),
            nn.ReLU(),
            nn.Linear(32,3)
        )

    def forward(self,x):
        return self.layer(x)


model = Classifier()
x = torch.randn(8, 10)
logits = model(x)

print(logits.shape)
# torch.Size([8, 3])

torch.Size([8, 3])


In [57]:
# Q6. What is the difference between torch.cat() and torch.stack()?

# Answer:
# torch.cat() concatenates tensors along an existing dimension. 
# torch.stack() creates a new dimension.

import torch

a = torch.tensor([1, 2])
b = torch.tensor([3, 4])

print(torch.cat([a, b]).shape)
# torch.Size([4])

print(torch.stack([a, b]).shape)
# torch.Size([2, 2])


# For cat, the tensors must have matching sizes in every dimension except the concatenation dimension.

# For stack, the tensors must have the same shape.

# Interview takeaway: Always check tensor dimensions before combining tensors.

torch.Size([4])
torch.Size([2, 2])


In [58]:
# Q7. What happens when you call .backward() twice?

import torch

x = torch.tensor(3.0, requires_grad=True)

y = x ** 2
y.backward()

print(x.grad)

y = x ** 2
y.backward()

print(x.grad)

tensor(6.)
tensor(12.)


In [59]:
# Q8. What is the difference between model.eval() and torch.no_grad()?

# They solve different problems.

model.eval()
with torch.no_grad():    
    predictions = model(inputs)


# - model.eval() changes the behavior of modules such as Dropout and BatchNorm.
# - torch.no_grad() prevents ordinary autograd graph recording inside its scope.
# Calling model.eval() does not disable gradient tracking.


# For inference-only code,we can also use:

model.eval()
with torch.inference_mode():    
    predictions = model(inputs)


# Interview takeaway: Evaluation mode controls layer behavior; 
# inference mode controls gradient-related tracking and can enable additional optimizations.

In [60]:
# Q9. Why might this training code fail?

import torch.nn as nn

model = nn.Linear(10, 3)
inputs = torch.randn(8, 10)
targets = torch.randint(0, 3, (8,))

criterion = nn.CrossEntropyLoss()

logits = model(inputs)
loss = criterion(logits, targets)


# Answer:

# This code is valid as written.
# - inputs has shape (8, 10): eight samples, ten features each.
# - logits has shape (8, 3): eight samples, three classes.
# - targets has shape (8,): one class index per sample.
# CrossEntropyLoss expects raw logits and, for this classification setup, integer class indices of dtype torch.long.

# A common mistake is applying softmax before passing logits to CrossEntropyLoss.

# Training loop:

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()

optimizer.zero_grad(set_to_none=True)
logits = model(inputs)
loss = criterion(logits, targets)
loss.backward()
optimizer.step()

# Interview takeaway: Understand the expected shapes and data types of both model outputs and targets.

In [ ]:
# Q10. How do you count trainable parameters in a PyTorch model?
# Answer:


import torch.nn as nn

model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 3)
)

total = sum(parameter.numel() for parameter in model.parameters())

trainable = sum( parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

print("Total parameters:", total)
print("Trainable parameters:", trainable)


# Explanation:

# The first linear layer contains \(10 \times 32 + 32 = 352\) parameters.

# The second contains \(32 \times 3 + 3 = 99\) parameters.

# Total: \(352 + 99 = 451\).

# The ReLU layer has no trainable parameters.

# Interview takeaway: Parameter counting helps estimate model size and identify which parameters are frozen during transfer learning.

Total parameters: 451
Trainable parameters: 451
